<a href="https://colab.research.google.com/github/keduog/EDU-AI-Training/blob/main/Day4/Session1/session1_resource_wall.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 1 — From One Neuron to the Resource Wall

**Day 4 · 09:00 – 10:30**

Today you will:
1. Watch **a single neuron** learn a rule nobody told it
2. Scale that idea up to a real language model and **prompt it**
3. Discover the wall: try to load **Meta's Llama-2-7B** and watch it fail

## Before you start
**Runtime → Change runtime type → T4 GPU → Save**

Do this now. Nothing below works properly without it.

---
# PART 1 — The Magic of a Single Neuron

## Cell 1 — A puzzle for you (no code yet)

Someone invented a new operation called **⭗**. Here is all you know about it:

```
4  ⭗  6  ⭗  8   =  100
5  ⭗  7  ⭗  9   =  110
6  ⭗  8  ⭗  10  =  120
8  ⭗  10 ⭗  12  =  140
```

**Questions:**
- What is `14 ⭗ 16 ⭗ 18` ?
- What is `22 ⭗ 24 ⭗ 26` ?

**Spend two minutes trying it by hand before running any code.**

Nobody told you the rule. You have only examples. That is *exactly* the situation
every machine learning model is in.

## Cell 2 — Now let one neuron figure it out

A single artificial neuron computes one simple thing:

```
output = w1·x1 + w2·x2 + w3·x3 + b
```

Four numbers: three **weights** and one **bias**. That is the entire model.

It starts with random values and knows nothing. We show it the four examples over and
over. Each time, it measures how wrong it is (**loss**) and nudges its four numbers
slightly in the direction that reduces the error (**backpropagation**).

> Think of loss as an overall **blame score**. Backpropagation works out who is
> responsible and shares the blame across every weight.

In [ ]:
import numpy as np

np.random.seed(42)

# The four examples - this is ALL the neuron ever sees
X = np.array([[4, 6, 8],
              [5, 7, 9],
              [6, 8, 10],
              [8, 10, 12]], dtype=float)
y = np.array([100, 110, 120, 140], dtype=float)

# The entire model: 3 weights + 1 bias, starting random
w = np.random.randn(3) * 0.1
b = 0.0
lr = 0.005          # learning rate - how big each nudge is

print("Before training, the neuron guesses:")
print(" ", np.round(X @ w + b, 2))
print("but the answers should be:")
print(" ", y)

## Cell 3 — Train it for 1500 epochs

Watch the **loss** column. That single number is the whole story.

In [ ]:
for epoch in range(1, 1501):
    pred = X @ w + b            # 1. guess
    err  = pred - y             # 2. how wrong?
    loss = np.mean(err ** 2)    # 3. blame score

    # 4. backpropagation: how much is each weight to blame?
    grad_w = 2 * X.T @ err / len(y)
    grad_b = 2 * err.mean()

    # 5. nudge each parameter against its blame
    w -= lr * grad_w
    b -= lr * grad_b

    if epoch in (1, 100, 500, 1000, 1500):
        print(f"epoch {epoch:>5}   loss {loss:>12.4f}   guesses {np.round(pred,1)}")

You should see something like:

```
epoch     1   loss   13830.9376   guesses [0.6 0.7 0.8 1. ]
epoch   100   loss      24.5328   guesses [ 94.4 107.8 121.1 147.8]
epoch   500   loss       2.0582   guesses [ 98.1 109.  120.  141.9]
epoch  1500   loss       0.0048   guesses [ 99.9 110.  120.  140.1]
```

**From 13,831 to 0.005.** Nobody told it the rule. It found the rule by being wrong
1,500 times and adjusting slightly each time.

## Cell 4 — The real test: questions it has NEVER seen

In [ ]:
tests = np.array([[14, 16, 18],
                  [22, 24, 26]], dtype=float)

for row in tests:
    answer = row @ w + b
    print(f"{int(row[0])} ⭗ {int(row[1])} ⭗ {int(row[2])}  =  {answer:.1f}")

print()
print("The correct answers are 200 and 280.")
print()
print("The neuron never saw these numbers during training.")
print("It learned the RULE, not the answers.")
print()
print("Its four learned numbers:")
print("  weights:", np.round(w, 2))
print("  bias   :", round(b, 2))

### That is machine learning, complete, in twenty lines

| Step | What happened |
|---|---|
| **Guess** | `X @ w + b` |
| **Measure error** | `loss = mean((pred - y)²)` |
| **Assign blame** | gradients — how much each weight contributed |
| **Nudge** | `w -= lr * grad` |
| **Repeat** | 1,500 times |

**Everything else in this course is this same loop, made bigger.**

## Cell 5 — Now imagine scaling that up

In [ ]:
one_neuron = 4                    # 3 weights + 1 bias

models = [
    ("Our neuron (today)",      one_neuron),
    ("A small image network",   1_000_000),
    ("BERT-base",             110_000_000),
    ("Qwen2-0.5B",            494_000_000),
    ("Llama-2-7B (Meta)",   6_740_000_000),
    ("Llama-3-70B (Meta)",  70_000_000_000),
    ("PaLM 540B (Day 1!)", 540_000_000_000),
]

print(f"{'Model':<24}{'parameters':>18}{'vs our neuron':>18}")
print("-" * 60)
for name, n in models:
    print(f"{name:<24}{n:>18,}{n//one_neuron:>17,}x")

**Qwen2-0.5B has 494 million of those numbers.** Same loop, same idea —
guess, measure blame, nudge — just repeated across half a billion parameters
and trillions of words instead of four examples.

That is why it can do things a single neuron cannot. And that is why it needs a GPU.

---
# PART 2 — Load a Real Model and Talk to It

## Cell 6 — Check your GPU first

In [ ]:
!nvidia-smi

In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("Name        :", p.name)
    print("Total memory: %.1f GB" % (p.total_memory / 1024**3))
else:
    print("NO GPU -> Runtime > Change runtime type > T4 GPU")

That memory number — about **15 GB** — is the wall. Remember it.

## Cell 7 — Install the libraries

In [ ]:
!pip install -q transformers accelerate
print("done")

## Cell 8 — Load Qwen2-0.5B

494 million parameters. Predicted memory: 494M × 2 bytes ≈ **0.9 GB**.

Let us see if the prediction holds.

In [ ]:
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2-0.5B-Instruct"

t0 = time.time()
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()

print("Loaded in %.1f seconds" % (time.time() - t0))

used  = torch.cuda.memory_allocated(0) / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print("Using %.2f GB of %.1f GB  (%.0f%% of the GPU)" % (used, total, used/total*100))

n_params = sum(p.numel() for p in model.parameters())
print("Parameters: %s" % f"{n_params:,}")

**Predicted 0.9 GB. Measured ~0.94 GB.** The formula works.

You have used about **6%** of the GPU. All that free space is what makes training
possible this afternoon.

## Cell 9 — A language model predicts the next word

That is genuinely all it does. Everything else emerges from doing it very well.

Let us test it with the exact sentence from the lecture.

In [ ]:
def complete(prompt, max_new_tokens=20):
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs,
                             max_new_tokens=max_new_tokens,
                             do_sample=False,               # greedy = most likely word
                             pad_token_id=tok.eos_token_id)
    return tok.decode(out[0], skip_special_tokens=True)


for prompt in [
    "The capital city of Ethiopia is",
    "The sky is",
    "Water boils at",
]:
    print(">>>", complete(prompt))
    print()

## Cell 10 — Look at the actual probabilities

The model does not "know" the answer. It produces a **probability for every word in
its vocabulary**, then picks one.

In [ ]:
import torch.nn.functional as F

prompt = "The capital city of Ethiopia is"
inputs = tok(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    logits = model(**inputs).logits

# probabilities for the NEXT token only
probs = F.softmax(logits[0, -1], dim=-1)
top = torch.topk(probs, 8)

print(f'Prompt: "{prompt} ___"')
print()
print(f"{'next word':<20}{'probability':>12}")
print("-" * 32)
for score, idx in zip(top.values, top.indices):
    word = tok.decode(idx).replace("\n", "\\n")
    bar = "#" * int(score.item() * 40)
    print(f"{repr(word):<20}{score.item():>11.1%}  {bar}")

**This is the whole magic, exposed.** No lookup table, no database — just a
probability distribution over the vocabulary, learned from reading enormous amounts
of text using the same guess-blame-nudge loop as our single neuron.

Try changing the prompt to Amharic and re-run. Notice how much less confident it is.
That is Session 2.

## Cell 11 — Chat properly (with the chat template)

In [ ]:
def ask(question, max_new_tokens=100):
    messages = [{"role": "user", "content": question}]
    text = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs,
                             max_new_tokens=max_new_tokens,
                             do_sample=True, temperature=0.7,
                             pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


print(ask("What is the capital of Ethiopia? Answer in one sentence."))
print()
print("--- now in Amharic ---")
print(ask("የኢትዮጵያ ዋና ከተማ ማን ናት?"))

Note how the Amharic answer is weaker. Hold that thought — Session 2 measures it.

---
# PART 3 — The Wall: Meta's Llama-2-7B

Qwen2-0.5B used 6% of the GPU. So why not use a bigger, better model?

Let us try **Llama-2-7B** — Meta's 7-billion-parameter model.

## Cell 12 — First, do the arithmetic (10 seconds)

In [ ]:
def gb(params, bytes_each):
    return params * bytes_each / 1024**3

TOTAL    = 15.0    # what nvidia-smi reports on a free Colab T4
OVERHEAD = 1.3     # CUDA context + kernels + minimum activations
USABLE   = TOTAL - OVERHEAD

models = [
    ("Qwen2-0.5B",            494_000_000),
    ("Llama-2-7B  (Meta)",  6_740_000_000),
    ("Llama-3-70B (Meta)",  70_000_000_000),
    ("PaLM 540B",          540_000_000_000),
]

print(f"{'Model':<22}{'load fp16':>11}{'LoRA':>10}{'full FT':>11}   verdict")
print("-" * 70)
for name, n in models:
    load = gb(n, 2)      # just the weights, half precision
    lora = gb(n, 2.5)    # weights + small adapters
    full = gb(n, 16)     # 4 weights + 4 gradients + 8 optimizer
    headroom = USABLE - load        # what is left after the weights
    if   full < USABLE:      v = "full FT fits"
    elif lora < USABLE:      v = "LoRA fits"
    elif headroom > 1.5:     v = "loads, cannot train"
    elif headroom > 0:       v = "TOO TIGHT -> crashes"
    else:                    v = "WILL NOT LOAD"
    print(f"{name:<22}{load:>9.1f}GB{lora:>8.1f}GB{full:>9.1f}GB   {v}")

print()
print(f"GPU reports        : {TOTAL:.1f} GB")
print(f"CUDA overhead      : {OVERHEAD:.1f} GB  (context, kernels, workspace)")
print(f"Actually usable    : {USABLE:.1f} GB  <-- this is the REAL wall")

### Read the last three lines carefully

The GPU *reports* 15 GB, but you never get all of it. CUDA itself reserves over a
gigabyte before your model loads a single weight.

**Llama-2-7B needs ~12.6 GB of weights against ~13.7 GB usable.** That leaves under
a gigabyte for activations, KV cache and workspace — which is not enough to run even
one short prompt.

This is why "it has 15 GB and the model is 12.6 GB, so it fits" is wrong.
Let us measure the real limit.

## Cell 13 — Load the REAL Llama-2 architecture (no 13 GB download)

Here is a useful trick. `init_empty_weights()` builds the genuine model architecture on
a "meta" device — **real layer shapes, real parameter count, zero memory used, and only
a tiny config file downloaded.**

We get the true numbers without waiting for a 13 GB download.

In [ ]:
from transformers import AutoConfig, AutoModelForCausalLM
from accelerate import init_empty_weights

LLAMA = "NousResearch/Llama-2-7b-hf"   # ungated mirror of Meta's Llama-2-7B

config = AutoConfig.from_pretrained(LLAMA)
print("Architecture   :", config.architectures)
print("Hidden size    :", config.hidden_size)
print("Layers         :", config.num_hidden_layers)
print("Attention heads:", config.num_attention_heads)
print("Vocab size     :", config.vocab_size)

# Build it with NO memory allocated
with init_empty_weights():
    skeleton = AutoModelForCausalLM.from_config(config)

n_llama = sum(p.numel() for p in skeleton.parameters())
print()
print(f"REAL parameter count: {n_llama:,}")

In [ ]:
needed_fp16 = n_llama * 2 / 1024**3
cuda_overhead = 0.8                      # CUDA context, kernels, workspace
activations   = 0.5                      # even one short prompt needs some

total_needed = needed_fp16 + cuda_overhead + activations
have = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"Weights (fp16)     : {needed_fp16:>6.2f} GB")
print(f"CUDA overhead      : {cuda_overhead:>6.2f} GB")
print(f"Activations        : {activations:>6.2f} GB")
print("                     -------")
print(f"TOTAL NEEDED       : {total_needed:>6.2f} GB")
print(f"YOU HAVE           : {have:>6.2f} GB")
print()
if total_needed > have:
    print(f">>> SHORT BY {total_needed - have:.2f} GB. This will crash. <<<")
else:
    print("It might just fit - but with no room to do anything useful.")

## Cell 14 — Now watch it actually crash

This allocates Llama-2-7B-sized memory on the GPU **for real**.

It will fail. That is the point of the cell.

In [ ]:
import gc

llama_needs = n_llama * 2 / 1024**3
print(f"Llama-2-7B needs {llama_needs:.2f} GB just for its weights.")
print("Let us find out how much this GPU will ACTUALLY give us.")
print()

chunks = []
CHUNK_GB = 0.5
allocated = 0.0

try:
    while allocated < 40:
        n = int(CHUNK_GB * 1024**3 / 2)          # fp16 = 2 bytes
        chunks.append(torch.empty(n, dtype=torch.float16, device="cuda"))
        allocated += CHUNK_GB
        marker = "  <-- Llama-2-7B needs this much" if abs(allocated - llama_needs) < 0.3 else ""
        print(f"  allocated {allocated:>5.1f} GB{marker}")

except torch.cuda.OutOfMemoryError as e:
    print()
    print(">>> CRASHED - torch.cuda.OutOfMemoryError <<<")
    print(f">>> The GPU refused at {allocated:.1f} GB <<<")
    print()
    print(str(e)[:400])
    print()
    if allocated < llama_needs:
        print(f"Llama-2-7B needs {llama_needs:.2f} GB. You got {allocated:.1f} GB.")
        print(f"SHORT BY {llama_needs - allocated:.2f} GB. It cannot load. Full stop.")
    else:
        print(f"You reached {allocated:.1f} GB, just past Llama's {llama_needs:.2f} GB of weights,")
        print("but with nothing left for activations - so a real forward pass still fails.")

except RuntimeError as e:
    print(">>> CRASHED <<<")
    print(str(e)[:400])

finally:
    del chunks
    gc.collect()
    torch.cuda.empty_cache()
    print()
    print("Memory released. Nothing is broken.")

### Read that error message carefully

You are looking for two numbers:
- `Tried to allocate ... GiB`
- `GPU ... has a total capacity of ... GiB of which ... is free`

**This is where most AI projects stop.** Not at the algorithm, not at the maths —
at the memory.

> ### The lesson
> The arithmetic in Cell 12 predicted this in **ten seconds**.
> Downloading Llama-2-7B first would have taken **15+ minutes** to reach the same
> conclusion. Always do the maths first.

## Cell 15 — Optional: try the genuine download (instructor demo)

Only run this if you have time and good bandwidth — it downloads about **13 GB**.

Set `REALLY_TRY = True` to run it.

In [ ]:
REALLY_TRY = False    # change to True only if you have time and bandwidth

if REALLY_TRY:
    try:
        print("Downloading Llama-2-7B... this can take 15+ minutes")
        big = AutoModelForCausalLM.from_pretrained(
            LLAMA,
            torch_dtype=torch.float16,
            device_map={"": 0},          # force everything onto the GPU
            low_cpu_mem_usage=True,
        )
        print("Loaded! Memory: %.2f GB" % (torch.cuda.memory_allocated(0)/1024**3))
    except Exception as e:
        print(">>> FAILED as predicted <<<")
        print(type(e).__name__)
        print(str(e)[:400])
    finally:
        gc.collect(); torch.cuda.empty_cache()
else:
    print("Skipped. Cell 14 already proved the point without the download.")

## Cell 16 — Clean up and check you are still fine

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

used = torch.cuda.memory_allocated(0) / 1024**3
print("GPU memory now in use: %.2f GB" % used)
print()
print("Qwen2-0.5B should still be loaded and working:")
print(complete("The capital city of Ethiopia is"))

> **If anything is broken:** `Runtime → Restart session`, then re-run from Cell 6.
> Nothing is permanently damaged — the GPU just needs clearing.

---

# What you learned

| Part | Idea |
|---|---|
| **One neuron** | Guess → measure blame → nudge → repeat. That is all learning is. |
| **Scaling up** | Qwen2-0.5B is the same loop across 494,000,000 parameters |
| **Prompting** | A language model outputs a probability for every possible next word |
| **The wall** | Llama-2-7B needs ~13.9 GB. You have 15 GB. It crashes. |
| **The formula** | `parameters × bytes` predicts in 10 seconds what a download proves in 15 minutes |

## The choice this forces

You cannot run Meta's 7B model here. You **can** run Qwen2-0.5B — and this afternoon
you will fine-tune it into something genuinely useful for Amharic.

**A small model you can actually train beats a big model you can only admire.**

**Next:** Session 2 — why this model struggles with Amharic, measured properly.